# Homework 4: Evaluation

Solving https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/04-evaluation/homework.md

## Setup

Reuse `rag_helper.py` and `evaluation_utils.py` (already downloaded into this folder) and load the course lesson pages from GitHub, pinned to commit `8c1834d`.

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()


In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
len(documents)


72

## Q1. Generating questions

Generate 5 questions each for the first 3 lesson pages, using the same structured-output approach (`Questions` model + `llm_structured`) as in the module, and average the input tokens across the 3 calls.

In [3]:
from pydantic import BaseModel
import json

from evaluation_utils import llm_structured


class Questions(BaseModel):
    questions: list[str]


data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()


In [4]:
target_filenames = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
]

docs_by_name = {d["filename"]: d for d in documents}
selected_docs = [docs_by_name[name] for name in target_filenames]

input_tokens_per_call = []

for doc in selected_docs:
    user_prompt = json.dumps(doc)
    result, usage = llm_structured(
        client,
        data_gen_instructions,
        user_prompt,
        Questions,
    )
    input_tokens_per_call.append(usage.input_tokens)
    print(f"{doc['filename']}: {usage.input_tokens} input tokens, {len(result.questions)} questions")

avg_input_tokens = sum(input_tokens_per_call) / len(input_tokens_per_call)
print(f"\nAverage input tokens across 3 calls: {avg_input_tokens:.2f}")


01-agentic-rag/lessons/01-intro.md: 1020 input tokens, 5 questions


01-agentic-rag/lessons/02-environment.md: 1286 input tokens, 5 questions


01-agentic-rag/lessons/03-rag.md: 1753 input tokens, 5 questions

Average input tokens across 3 calls: 1353.00


**Answer Q1:** the average lands in the low thousands, closest to **1400**.

## The full ground truth

Load the pre-generated 360-question ground truth (already downloaded as `ground-truth.csv`).

In [5]:
import pandas as pd

ground_truth = pd.read_csv("ground-truth.csv").to_dict(orient="records")
len(ground_truth), ground_truth[0]


(360,
 {'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
  'filename': '01-agentic-rag/lessons/01-intro.md'})

## Searching the chunks

Chunk the 72 pages, then build a text index (`Index`) and a vector index (`VectorSearch`) from minsearch, both keyed on `filename`, matching the approach from homework 2.

In [6]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)


295

In [7]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
text_index.fit(chunks)


In [8]:
from sentence_transformers import SentenceTransformer
from minsearch import VectorSearch
import numpy as np

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = [c["content"] for c in chunks]
X = embed_model.encode(chunk_texts, show_progress_bar=True, convert_to_numpy=True)
X = np.asarray(X)

vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(X, chunks)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

In [9]:
def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)


def vector_search(query, num_results=5):
    query_vector = embed_model.encode(query, convert_to_numpy=True)
    return vector_index.search(query_vector, num_results=num_results)


## Q2. First result with text search

In [10]:
q = ground_truth[0]["question"]
q


"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [11]:
text_results = text_search(q)
first_text_filename = text_results[0]["filename"]
first_text_filename


'01-agentic-rag/lessons/03-rag.md'

## Q3. First result with vector search

In [12]:
vector_results = vector_search(q)
first_vector_filename = vector_results[0]["filename"]
first_vector_filename


'01-agentic-rag/lessons/01-intro.md'

This question was generated from `01-agentic-rag/lessons/01-intro.md`. Notice that one method finds the right page at the top and the other doesn't — that's exactly why we measure across the whole dataset instead of trusting one query.

## Evaluation metrics

Reuse the same `compute_relevance` / `hit_rate` / `mrr` / `evaluate` functions from the lecture, but key relevance on `filename` instead of `id`.

In [13]:
from tqdm.auto import tqdm


def compute_relevance(q, search_function):
    filename = q["filename"]
    results = search_function(query=q["question"])
    return [int(d["filename"] == filename) for d in results]


def compute_relevance_total(ground_truth, search_function):
    relevance_total = []
    for q in tqdm(ground_truth):
        relevance_total.append(compute_relevance(q, search_function))
    return relevance_total


def hit_rate(relevance):
    cnt = 0
    for line in relevance:
        if 1 in line:
            cnt = cnt + 1
    return cnt / len(relevance)


def mrr(relevance):
    total_score = 0.0
    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break
    return total_score / len(relevance)


def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }


## Q4. Evaluating text search

In [14]:
text_metrics = evaluate(ground_truth, text_search)
text_metrics


  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

## Q5. Evaluating vector search

In [15]:
vector_metrics = evaluate(ground_truth, vector_search)
vector_metrics


  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.8083333333333333, 'mrr': 0.6356944444444446}

## Q6. Tuning hybrid search

In [16]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


def hybrid_search(query, k=60, num_results=5):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k, num_results=num_results)


In [17]:
hybrid_mrr_by_k = {}

for k in [1, 50, 100, 200]:
    search_fn = lambda query, k=k: hybrid_search(query, k=k)
    metrics = evaluate(ground_truth, search_fn)
    hybrid_mrr_by_k[k] = metrics["mrr"]
    print(f"k={k}: hit_rate={metrics['hit_rate']:.4f}, mrr={metrics['mrr']:.4f}")

best_k = max(hybrid_mrr_by_k, key=lambda k: (hybrid_mrr_by_k[k], -k))
print(f"\nBest k: {best_k} (MRR={hybrid_mrr_by_k[best_k]:.4f})")


  0%|          | 0/360 [00:00<?, ?it/s]

k=1: hit_rate=0.8583, mrr=0.6723


  0%|          | 0/360 [00:00<?, ?it/s]

k=50: hit_rate=0.8472, mrr=0.6721


  0%|          | 0/360 [00:00<?, ?it/s]

k=100: hit_rate=0.8472, mrr=0.6721


  0%|          | 0/360 [00:00<?, ?it/s]

k=200: hit_rate=0.8472, mrr=0.6721

Best k: 1 (MRR=0.6723)


## Summary of answers

| Question | Computed value | Closest option |
|---|---|---|
| Q1. Avg input tokens (3 calls) | 1353.00 | **1400** |
| Q2. Top text_search filename | `01-agentic-rag/lessons/03-rag.md` | **`01-agentic-rag/lessons/03-rag.md`** |
| Q3. Top vector_search filename | `01-agentic-rag/lessons/01-intro.md` | **`01-agentic-rag/lessons/01-intro.md`** |
| Q4. Hit Rate (text_search) | 0.7583 | **0.76** |
| Q5. MRR (vector_search) | 0.6357 | **0.65** |
| Q6. Best RRF `k` (by MRR) | k=1 → 0.6723 (vs. 0.6721 for 50/100/200) | **1** |
